In [1]:
import requests
from requests.auth import HTTPBasicAuth
import getpass
import json

email = input('email: ')
password = getpass.getpass('password: ')
url = 'http://www.vspdb.org/users/login'
headers = {}
headers['User-Agent'] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) 134.0.6998.118 Safari/537.36"
headers['Accept'] = "application/json"
r = requests.get(url, headers=headers, auth=HTTPBasicAuth(email,password))
token = json.loads(r.text)['token']

In [2]:
import pandas as pd
headers = {}
headers["Accept"] = "application/json"
headers["User-Agent"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) 134.0.6998.118 Safari/537.36"
headers["Authorization"] = "Bearer {}".format(token)
url = 'http://www.vspdb.org/tests?limit=100000'
r= requests.get(url, headers=headers)
test = pd.DataFrame.from_dict(json.loads(r.text))

In [3]:
url = 'http://www.vspdb.org/velocity-metadata?limit=100000'
r= requests.get(url, headers=headers)
velocity_meta = pd.DataFrame.from_dict(json.loads(r.text))

In [4]:
url = 'http://www.vspdb.org/hvsrMetadata?limit=100000'
r= requests.get(url, headers=headers)
hvsr_meta = pd.DataFrame.from_dict(json.loads(r.text))

In [6]:
hvsr_meta.head()

,id,test_id,hvsr_name,hvsr_datetime,hvsr_type,window_rejection,hvsr_number_of_mean_ordinates,hvsr_sensor_json,hvsr_processing_json,hvsr_comment
0,1,2777,AZ.PFO,2011-09-16,False,None,500,"{""field_crew"": ""Carpenter/ Collins"", ""weather""...","{""time_window_length"": 200.0, ""window_overlap""...",None
1,2,2778,AZ.PFO,2011-09-16,False,None,500,"{""field_crew"": ""Carpenter/ Collins"", ""weather""...","{""time_window_length"": 200.0, ""window_overlap""...",None
2,3,2779,AZ.PFO,2011-09-16,False,None,500,"{""field_crew"": ""Carpenter/ Collins"", ""weather""...","{""time_window_length"": 200.0, ""window_overlap""...",None
3,4,2780,BK.KCC,2010-10-21,False,None,500,"{""field_crew"": ""Martin/Carter"", ""weather"": ""wi...","{""time_window_length"": 200.0, ""window_overlap""...",None
4,5,2781,BK.KCC,2010-10-21,False,None,500,"{""field_crew"": ""Martin/Carter"", ""weather"": ""wi...","{""time_window_length"": 200.0, ""window_overlap""...",None


In [11]:
test

,test_id,site_id,user_id,citation_id,test_name,test_latitude,test_longitude,test_elevation,test_comments
0,1,1,13,138,VAR,35.92080,-120.45100,567.000,None
1,2,2,13,129,PNF,33.61170,-116.45940,1280.000,None
2,3,3,13,129,KNW,33.71110,-116.71330,1469.000,None
3,4,4,13,120,MCG,37.54820,-118.81270,2647.000,None
4,5,5,13,121,CLD,37.11750,-121.55160,240.000,None
...,...,...,...,...,...,...,...,...,...
7863,7912,236,2341,754,99_SSHV,37.82814,-122.37511,2.272,None
7864,7913,236,2341,754,100_SSHV,37.82829,-122.37182,2.947,None
7865,7914,236,2341,754,101_SSHV,37.82743,-122.37023,2.882,None
7866,7915,236,2341,754,102_SSHV,37.82274,-122.36546,3.499,None


In [7]:
test.rename(columns={'id': 'test_id'}, inplace=True)

In [12]:
# merge test
merged_hvsr_df = pd.merge(test, hvsr_meta, on='test_id', how='inner')
merged_hvsr_df.head()
merged_hvsr_df.to_csv('merged_hvsr_df.csv', index=False)

In [13]:
merged_velocity_df = pd.merge(test, velocity_meta, on='test_id', how='inner')
merged_velocity_df.head()
merged_velocity_df.to_csv('merged_velocity_df.csv', index=False)

In [48]:
url = 'http://www.vspdb.org/travelTimeMetadata?limit=1000000'
r= requests.get(url, headers=headers)
traveltime_meta = pd.DataFrame.from_dict(json.loads(r.text))

In [49]:
url = 'http://www.vspdb.org/travelTimeData?limit=100000000'
r= requests.get(url, headers=headers)
traveltime_data = pd.DataFrame.from_dict(json.loads(r.text))

In [50]:
url = 'http://www.vspdb.org/cptMetadata?limit=100000000'
r= requests.get(url, headers=headers)
cpt_meta = pd.DataFrame.from_dict(json.loads(r.text))

In [51]:
url = 'https://vspdb.org/cptData?where=cpt_metadata_id<800&limit=100000000'
r= requests.get(url, headers=headers)
cpt_data1 = pd.DataFrame.from_dict(json.loads(r.text))

In [52]:
url = 'https://vspdb.org/cptData?where=cpt_metadata_id>=800&limit=100000000'
r= requests.get(url, headers=headers)
cpt_data2 = pd.DataFrame.from_dict(json.loads(r.text))

In [53]:
cpt_data = pd.concat([cpt_data1, cpt_data2], ignore_index=True)

In [54]:
test.to_csv("test_table.csv", index=False)
traveltime_meta.to_csv("traveltime_meta_table.csv", index=False)
traveltime_data.to_csv("traveltime_data_table.csv", index=False)
cpt_data.to_csv("cpt_data_table.csv", index=False)
cpt_meta.to_csv("cpt_meta_table.csv", index=False)